## 차원 축소란?

**차원 축소(Dimensionality Reduction)** 는 비지도 학습(Unsupervised Learning)의 한 분류인 **변환(Transformation)** 계열의 알고리즘입니다.

- 수백 개의 피처(feature)를 가진 데이터셋을 **소수의 차원으로 근사(approximate)** 하여 표현
- 주요 활용 목적: **데이터 탐색(EDA), 시각화, 군집 패턴 파악**
- 예: 200개의 피처 → 2개의 차원으로 축소 후 2D 산점도(scatterplot)로 시각화

---

## 주요 차원 축소 알고리즘 3가지

| 알고리즘 | 유형 | 특징 |
|---|---|---|
| **PCA** (Principal Component Analysis) | 선형 변환 | 분산을 최대화하는 방향으로 회전, 빠르고 범용적 |
| **MDS** (Multi-Dimensional Scaling) | 다양체 학습 | 원본 공간의 거리 관계를 저차원에서 보존 |
| **t-SNE** (t-distributed Stochastic Neighbor Embedding) | 다양체 학습 | 이웃 포인트 간 거리 보존에 특화, 지역 구조가 뚜렷한 데이터에 강점 |

---

## 1단계. 라이브러리 및 데이터 로드

In [ ]:
# 필수 라이브러리 임포트
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.datasets import load_breast_cancer  # sklearn 내장 유방암 데이터셋

# 유방암 데이터셋 로드
cancer = load_breast_cancer()
(X_canc, y_canc) = load_breast_cancer(return_X_y=True)

---

## 2단계. 헬퍼 함수 정의

레이블이 있는 2D 산점도를 그리기 위한 범용 함수입니다.  
차원 축소 후 결과를 시각화할 때 반복 사용됩니다.

In [ ]:
def plot_labelled_scatter(X, y, class_labels, s):
    """
    레이블된 산점도를 그리는 헬퍼 함수

    Parameters:
        X            : 2D 배열 형태의 피처 데이터 (n_samples, 2)
        y            : 정수형 레이블 배열
        class_labels : 클래스 이름 리스트 (예: ['malignant', 'benign'])
        s            : 그림 크기 튜플 (figsize)
    """
    num_labels = len(class_labels)

    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    marker_array = ['o', '^', '*']
    color_array = ['#FFFF00', '#00AAFF', '#000000', '#FF00AA']
    cmap_bold = ListedColormap(color_array)
    bnorm = BoundaryNorm(np.arange(0, num_labels + 1, 1), ncolors=num_labels)
    plt.figure(figsize=s)

    plt.scatter(X[:, 0], X[:, 1], s=80, c=y, cmap=cmap_bold, norm=bnorm,
                alpha=0.4, edgecolor='black', lw=1)

    # 테두리 정리 (위/오른쪽 축 제거)
    sp = plt.gca().spines
    sp['top'].set_visible(False)
    sp['right'].set_visible(False)

    plt.grid(which='both', color='lightslategrey', alpha=0.3)

    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)

    # 범례 생성
    h = []
    for c in range(0, num_labels):
        h.append(mpatches.Patch(color=color_array[c], label=class_labels[c]))
    plt.legend(handles=h, fontsize=15, frameon=False)

---

# 1. 주성분 분석 (PCA, Principal Component Analysis)

## 개념 정리

### PCA란?
PCA는 데이터의 **분산(Variance)** 을 최대한 보존하는 방향으로 좌표계를 회전시키는 **선형 변환** 기법입니다.

### 핵심 원리
1. 원본 데이터 포인트들의 분포를 바라볼 때, 가장 분산이 큰 방향 → **제1 주성분 (PC1)**
2. PC1에 **직교(orthogonal)** 하며, 그 다음으로 분산이 큰 방향 → **제2 주성분 (PC2)**
3. 이 과정을 원하는 차원 수만큼 반복

## Step 1-1. 데이터 정규화 및 PCA 적용

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# PCA 적용 전: 각 피처를 평균=0, 분산=1로 표준화 (StandardScaler)
canc_norm = StandardScaler().fit(X_canc).transform(X_canc)

# PCA 객체 생성 및 피팅 (2개의 주성분으로 축소)
pca = PCA(n_components=2).fit(canc_norm)

# 변환 적용
canc_pca = pca.transform(canc_norm)

print('Number of Features in Breast Cancer DataSet Before PCA : {}\n\n'
      'Number of Features in Breast Cancer DataSet After PCA  : {}'
      .format(X_canc.shape[1], canc_pca.shape[1]))

## Step 1-2. PCA 결과 시각화

In [ ]:
# 헬퍼 함수로 PCA 변환된 데이터 산점도 시각화
plot_labelled_scatter(canc_pca, y_canc, ['malignant', 'benign'], (15, 9))

plt.xlabel('First principal component', fontsize=15)
plt.ylabel('Second principal component', fontsize=15)
plt.title('Breast Cancer Dataset PCA (n_components = 2)', fontsize=17);

###  결과 해석
- 30개의 피처가 **2개의 주성분**으로 축소됨
- 시각화 결과: `malignant(악성)`과 `benign(양성)` 두 클래스가 **명확히 분리**됨
- 단순한 **로지스틱 회귀**로도 쉽게 분류 가능한 수준의 분리도

## Step 1-3. 피처 간 상관관계 히트맵 (PCA components_ 활용)

In [ ]:
# pca.components_: 각 주성분과 원본 피처 간의 상관 계수 행렬
fig = plt.figure(figsize=(20, 9))
plt.imshow(pca.components_, interpolation='none', cmap='viridis')

feature_names = list(cancer.feature_names)

plt.gca().set_xticks(np.arange(len(feature_names)))
plt.gca().set_yticks(np.arange(2))
plt.gca().set_xticklabels(feature_names, rotation=90, fontsize=14)
plt.gca().set_yticklabels(['First PC', 'Second PC'], fontsize=14)

plt.colorbar(orientation='horizontal',
             ticks=[pca.components_.min(), 0, pca.components_.max()],
             pad=0.5);

###  결과 해석

- **PC1 (첫 번째 주성분)**: 30개 피처 모두 **양수(positive)** 방향 → 모든 피처가 함께 증가/감소하는 경향 (전반적 양의 상관관계)
- **PC2 (두 번째 주성분)**: 일부 피처는 양수, 일부는 **음수(negative)** 방향
  - `mean texture` ↔ `worst texture`, `mean radius` ↔ `worst radius` 쌍이 같은 방향으로 함께 변화
  - 이들은 나머지 피처들과 반대 방향으로 변화하는 군집 형성

---

# 2. 다차원 척도법 (MDS, Multi-Dimensional Scaling)

## 개념 정리

### MDS란?
MDS는 **다양체 학습(Manifold Learning)** 알고리즘의 일종으로, 고차원 데이터 내에 존재하는 **저차원 구조(manifold)** 를 발견하는 데 특화되어 있습니다.

### 다양체(Manifold)란?
고차원 공간 안에 존재하는 저차원 곡면 구조입니다.  
대표적인 예: **Swiss Roll Dataset** — 3D 공간 안에 2D 시트가 말려 있는 형태

> PCA는 선형 변환이므로 이런 비선형 구조를 발견하지 못합니다.

### MDS의 핵심 목표
- 원본 고차원 공간에서 **포인트 간 거리(distance)** 를 최대한 보존하면서 저차원(주로 2D)으로 투영
- 결과적으로 데이터의 **군집 패턴**을 저차원에서도 확인 가능

## Step 2-1. MDS 적용 및 시각화

In [ ]:
from sklearn.manifold import MDS

# MDS 객체 생성 (2차원으로 축소, random_state 고정)
mds = MDS(n_components=2, random_state=2)

# 표준화된 데이터에 MDS 적용 (fit + transform 동시 수행)
canc_mds = mds.fit_transform(canc_norm)

print('Number of Features in Breast Cancer DataSet Before MDS : {}\n\n'
      'Number of Features in Breast Cancer DataSet After MDS  : {}'
      .format(X_canc.shape[1], canc_mds.shape[1]))

# 시각화
plot_labelled_scatter(canc_mds, y_canc, ['malignant', 'benign'], (15, 9))

plt.xlabel('First MDS dimension', fontsize=15)
plt.ylabel('Second MDS dimension', fontsize=15)
plt.title('Breast Cancer Dataset MDS (n_components = 2)', fontsize=17);

### 결과 해석
- MDS도 두 클래스를 잘 분리하여 시각화
- PCA 결과와 **분포 패턴이 다름**: 두 알고리즘이 서로 다른 수학적 원리로 작동하기 때문
  - PCA: 분산 최대화 기반 선형 회전
  - MDS: 거리 보존 기반 비선형 변환

---

# 3. t-분포 확률적 이웃 임베딩 (t-SNE)

## 개념 정리

### t-SNE란?
t-SNE는 데이터 시각화를 위한 강력한 다양체 학습 알고리즘입니다.

### 핵심 목표
- 고차원 데이터에서 **이웃(neighbor)** 포인트 간 거리 관계를 2D 공간에서 최대한 보존
- 특히 **가까운 포인트 간의 거리** 보존에 더 큰 가중치 부여
- 전역적 구조보다 **지역적 구조(local structure)** 보존에 특화

## Step 3-1. t-SNE 적용 및 시각화

In [ ]:
from sklearn.manifold import TSNE

# t-SNE 객체 생성 (기본 n_components=2, random_state 고정)
tsne = TSNE(random_state=42)

# 표준화된 데이터에 t-SNE 적용
canc_tsne = tsne.fit_transform(canc_norm)

print('Number of Features in Breast Cancer DataSet Before T-SNE : {}\n\n'
      'Number of Features in Breast Cancer DataSet After T-SNE  : {}'
      .format(X_canc.shape[1], canc_tsne.shape[1]))

# 시각화
plot_labelled_scatter(canc_tsne, y_canc, ['malignant', 'benign'], (15, 9))

plt.xlabel('First T-SNE Dimension', fontsize=14)
plt.ylabel('Second T-SNE Dimension', fontsize=14)
plt.title('Breast Cancer Dataset T-SNE', fontsize=17);

### 결과 해석
- t-SNE도 두 클래스를 잘 분리하여 시각화
- 유방암 데이터셋처럼 **지역 구조가 뚜렷한 데이터**에서 효과적
- PCA, MDS와 분포 형태가 다르지만 클래스 분리는 동일하게 성공